# YORVYN — Reproducible Google Colab GPU Execution & Worker

This notebook provides a reproducible, version-pinned Google Colab GPU environment for YORVYN.

### Security & Reproducibility Invariants
1. **No Embedded Tokens:** GitHub PATs and worker tokens are loaded ephemerally via `google.colab.userdata` or `getpass` and are never written to git config, files, or logs.
2. **Exact Version Pinning:** Every run checks out an explicit `TARGET_BRANCH` or `TARGET_COMMIT_SHA` (never assuming `main` is correct for an experiment).
3. **Pinned Dependencies:** Installs from `backend/requirements.txt` and logs exact Python, PyTorch, and library versions.
4. **Full Provenance:** Records Git repo, branch, commit SHA, GPU model, checkpoint SHA-256, dataset version, parameters, output artifact SHA-256s, and execution status via `scripts/ops/colab_experiment_runner.py`.

In [ ]:
# Step 1: Select Exact Code Version & Persistent Storage Target
TARGET_REPO_OWNER = "anujsandhu"
TARGET_REPO_NAME = "Yorvyn"
TARGET_BRANCH = "release/stabilize-17.4"  # Set explicitly per experiment
TARGET_COMMIT_SHA = ""  # Optional: pin exact 40-char SHA; overrides branch head if set
DATASET_VERSION = "yorvyn_golden_v1"
EXPERIMENT_ID = "colab_gpu_run_001"
MOUNT_GOOGLE_DRIVE = True

print(f"Configured target: {TARGET_REPO_OWNER}/{TARGET_REPO_NAME} @ {TARGET_COMMIT_SHA or TARGET_BRANCH}")

In [ ]:
# Step 2: Ephemeral GitHub Authentication & Exact-Version Clone
import getpass
import os
import subprocess
from pathlib import Path

try:
    from google.colab import userdata
    _gh_token = userdata.get("GITHUB_TOKEN")
except Exception:
    _gh_token = None

if not _gh_token:
    _gh_token = getpass.getpass("Enter GitHub Fine-Grained PAT (repo read-only, ephemeral): ").strip()

REPO_DIR = Path("/content/Yorvyn")
if not REPO_DIR.exists():
    # Use GIT_ASKPASS so the token never appears in .git/config or shell history
    askpass = Path("/tmp/yorvyn_git_askpass.sh")
    askpass.write_text("#!/bin/sh\ncase \"$1\" in\n  Username*) echo x-access-token ;;\n  Password*) echo \"$YORVYN_GH_TOKEN\" ;;\nesac\n")
    askpass.chmod(0o700)
    env = {**os.environ, "GIT_ASKPASS": str(askpass), "YORVYN_GH_TOKEN": _gh_token, "GIT_TERMINAL_PROMPT": "0"}
    subprocess.run(
        ["git", "clone", "--branch", TARGET_BRANCH, f"https://github.com/{TARGET_REPO_OWNER}/{TARGET_REPO_NAME}.git", str(REPO_DIR)],
        env=env,
        check=True,
    )
    askpass.unlink(missing_ok=True)

os.chdir(REPO_DIR)
if TARGET_COMMIT_SHA:
    subprocess.run(["git", "checkout", "--detach", TARGET_COMMIT_SHA], check=True)
else:
    subprocess.run(["git", "checkout", TARGET_BRANCH], check=True)

ACTIVE_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"Checked out exact commit SHA: {ACTIVE_SHA}")
del _gh_token

In [ ]:
# Step 3: Install Pinned Dependencies & Detect GPU
!pip install -q -r backend/requirements.txt

import sys
import torch

print("=" * 60)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
print("=" * 60)

In [ ]:
# Step 4: Persistent Storage Setup & Checkpoint Verification
import json
from pathlib import Path

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ARTIFACT_ROOT = Path("/content/drive/MyDrive/yorvyn_artifacts")
    except Exception:
        ARTIFACT_ROOT = Path("/content/yorvyn_artifacts")
else:
    ARTIFACT_ROOT = Path("/content/yorvyn_artifacts")

RUN_DIR = ARTIFACT_ROOT / EXPERIMENT_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = Path("models/checkpoints/checkpoint_manifest.json")
if manifest_path.exists():
    ckpt_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(f"Loaded checkpoint manifest entries: {len(ckpt_manifest)}")
else:
    print("No local checkpoint_manifest.json found; place verified checkpoints in Drive/object storage.")
print(f"Artifact export directory: {RUN_DIR}")

In [ ]:
# Step 5: Execute Selected YORVYN Module & Record Reproducibility Manifest
import sys
sys.path.insert(0, str(Path(".").resolve()))
sys.path.insert(0, str(Path("backend").resolve()))

from scripts.ops.colab_experiment_runner import build_experiment_manifest, write_manifest
from backend.app.services.garments.mesh_validation_service import MeshValidationService

# Example module execution: verify MeshValidationService on runtime environment
validator = MeshValidationService()
log_file = RUN_DIR / "execution.log"
log_file.write_text(
    f"experiment_id={EXPERIMENT_ID}\ncommit_sha={ACTIVE_SHA}\nvalidator={type(validator).__name__}\nstatus=SUCCEEDED\n",
    encoding="utf-8",
)

manifest = build_experiment_manifest(
    repo_root=Path(".").resolve(),
    experiment_id=EXPERIMENT_ID,
    dataset_version=DATASET_VERSION,
    execution_parameters={
        "target_branch": TARGET_BRANCH,
        "target_commit_sha": TARGET_COMMIT_SHA or ACTIVE_SHA,
        "module": "MeshValidationService",
    },
    checkpoint_path=None,
    output_artifact_paths=[log_file],
    execution_status="SUCCEEDED",
)
manifest_out = write_manifest(manifest, RUN_DIR / "reproducibility_manifest.json")
print(f"Saved reproducibility manifest to {manifest_out}")
print(manifest_out.read_text(encoding="utf-8"))